# 作业 01：验证 RTX 5060 能运行 PyTorch 训练

先按 [README](README.md) 配好环境，选择 **Python (cv_gpu)** 内核，再依次运行下面三格。你将检查 CUDA、运行一次 GPU 矩阵乘法，并在 GPU 上拟合直线 `y = 3x - 2`。运行成功后，保存包含输出的 Notebook。

In [1]:
# 1. 检查 Notebook 当前的 Python、PyTorch 和显卡
import sys

print(f"Python 路径：{sys.executable}")

import torch

print(f"PyTorch 版本：{torch.__version__}")
print(f"PyTorch 的 CUDA 版本：{torch.version.cuda}")
assert torch.version.cuda is not None, "当前 PyTorch 不含 CUDA，请检查内核和安装命令。"
assert torch.cuda.is_available(), "CUDA 不可用，请按 README 检查显卡驱动。"

device = torch.device("cuda:0")
print(f"显卡：{torch.cuda.get_device_name(device)}")
print(f"计算能力：{torch.cuda.get_device_capability(device)}")

Python 路径：C:\ProgramData\miniconda3\python.exe
PyTorch 版本：2.14.0+cu130
PyTorch 的 CUDA 版本：13.0
显卡：NVIDIA GeForce RTX 5060
计算能力：(12, 0)


In [2]:
# 2. 在 GPU 上运行矩阵乘法
matrix = torch.ones(256, 256, device=device)
product = matrix @ matrix

print(f"结果所在设备：{product.device}")
print(f"左上角的值：{product[0, 0].item():.0f}")
assert product.device == device
assert product[0, 0].item() == 256.0

结果所在设备：cuda:0
左上角的值：256


In [3]:
# 3. 在 GPU 上训练线性回归模型 y = 3x - 2
from torch import nn

# 生成训练数据，再将模型移到 GPU。优化器在模型就位后创建。
x = torch.linspace(-1, 1, 256, device=device).reshape(-1, 1)
y = 3 * x - 2
model = nn.Linear(1, 1).to(device)
with torch.no_grad():
    model.weight.zero_()
    model.bias.zero_()

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)

with torch.no_grad():
    initial_loss = criterion(model(x), y).item()

for _ in range(40):
    optimizer.zero_grad(set_to_none=True)
    loss = criterion(model(x), y)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    final_loss = criterion(model(x), y).item()

gradient = model.weight.grad
assert gradient is not None and gradient.device == device
assert x.device == y.device == model.weight.device == device
assert final_loss < initial_loss * 0.01, "训练后损失应明显下降。"

print(f"数据 / 模型参数 / 梯度：{x.device} / {model.weight.device} / {gradient.device}")
print(f"训练前损失：{initial_loss:.6f}")
print(f"训练后损失：{final_loss:.6f}")
print(f"学到的斜率：{model.weight.item():.3f}，截距：{model.bias.item():.3f}")

数据 / 模型参数 / 梯度：cuda:0 / cuda:0 / cuda:0
训练前损失：7.023529
训练后损失：0.000029
学到的斜率：2.991，截距：-2.000
